# 08 · Pangolin — an independent splice model

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID 35449021,
[github.com/tkzeng/Pangolin](https://github.com/tkzeng/Pangolin)) is a deep-learning
splice predictor trained on splice-site *usage* across tissues and species. It reports
**one 0-1 score** per variant rather than SpliceAI's four deltas, and uses the **same**
0.5 / 0.2 thresholds. Because the two models were built independently, **agreement
between SpliceAI and Pangolin** is stronger evidence than either alone — and §3 below
measures that agreement instead of asserting it.

That single score is convenient but lossy: it is the larger of a gain and a loss
magnitude, so it tells you *how much* splicing changes and not *which direction*. §2
spells out what that costs you relative to tools/07.

> ✅ **REAL.** Pangolin has no precomputed release and is not in dbNSFP, but
> **the build cell below runs the actual model locally** — weights ship inside the pip
> package, and it needs only the ~215 kb CFTR reference region (no whole-genome download).
> The default scope now scores **every CFTR2 variant with GRCh38 coordinates (~1,892 of
> 2,097), SNVs *and* indels**, so `source='REAL'`. Validated against real SpliceAI on the
> canonical alleles (e.g. c.2988+1G>A: Pangolin 0.86 vs SpliceAI 0.99).
>
> `SCOPE = "curated"` in the build cell still scores just the 5 classic splice alleles and stays
> `source='DEMO'` — the label follows the **coverage**, never the model.
>
> **Version:** there is no file to date-stamp here — the artefact is a *model run*, so the
> build cell records what a rerun would have to match (pangolin package version, the SHA-256
> of the twelve weight files it loaded, the reference region, and the torch/device it ran on)
> into `data/pangolin_cftr.release.json`, exposed as the `pangolin_release` column.

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · Pangolin — a newer splice model, one tidy score

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID **35449021**) was trained on
splice-site *usage* measured across multiple tissues and species, and tends to be a
little more sensitive than SpliceAI for some variant classes. For our purposes the nice
thing is that it returns a single 0-1 score (larger of the biggest splice-usage gain and
loss), directly comparable to SpliceAI's DS_max at the same 0.5 / 0.2 cut-points.

## 2 · REAL Pangolin scores — run the model, don't download a table

Unlike SpliceAI there is no precomputed file to download; you **run the model**. It is
lighter than it sounds — no whole-genome FASTA required, and a GPU only makes it faster.
See the build cell below for the exact install command and code.

`tk.load_pangolin()` reads whatever the build cell wrote. The default scope covers every
CFTR2 variant that has GRCh38 coordinates — ~1,892 of 2,097 — SNVs and indels alike.

**Why this reaches further than SpliceAI here.** Pangolin run locally pads its delta
track by the ref/alt length difference, so it scores indels as readily as SNVs. Of the
**780** CFTR2 variants that are not plain SNVs, **576** carry GRCh38 coordinates and
**575** of those get a Pangolin score — the one refusal is a single event larger than
the ±50 bp window can speak to. The precomputed SpliceAI extract in tools/07 reaches
**417** of the same 576.

Reach is not the same as usefulness, though. Most of those extra rows are frameshifts
and large deletions, and a splice verdict on a frameshift is usually a correct "no splice
impact" that tells you nothing about why the variant causes disease. The gain is that the
row exists and says 0.02 rather than being absent — a stated negative, not a silence.

**What the single score does not tell you.** SpliceAI keeps four deltas, so a 0.9 there
distinguishes "a real donor is being destroyed" (DS_DL) from "a cryptic acceptor is being
created" (DS_AG). Pangolin's model produces the same kind of directional information — a
largest *increase* and a largest *decrease* in predicted splice-site usage — but this
extract stores only `max(gain, |loss|)`, so the direction is discarded before it reaches
the CSV. Recovering it means re-running the model, not re-reading the file. When you need
the mechanism rather than the magnitude, read tools/07's deltas.

### Running the model — the build cell below

Pangolin has no per-gene download and no bulk file to filter — the only way to
get real scores is to **run the model locally**. That needs one extra install
and one small manual download (a reference sequence slice, not a raw-data file):

```bash
pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils torch
```

The model weights ship inside that pip package (no separate download), and the
cell below auto-fetches the ~215 kb CFTR reference region from Ensembl the
first time it runs, caching it at `data/cftr_region_grch38.fa` — **no
whole-genome FASTA needed**. It reads authoritative GRCh38 coordinates from
`data/cftr2_cftr.csv` (built in benchmark/01 — run that notebook first),
not hand-entered ones, so Pangolin scores the variant it is actually supposed
to. `SCOPE = "cftr2"` (default, below) scores every CFTR2 variant with GRCh38
coordinates — ~1,892 of 2,097, SNVs *and* indels, ~4 min on a GPU, `source='REAL'`;
`SCOPE = "curated"` scores only the 5 classic splice alleles and stays
`source='DEMO'` — **the label follows coverage, never the model.**

Variants that cannot be scored (no coordinates, or an indel bigger than the
±50 bp aggregation window can speak to) are kept with an empty score and a
`skip_reason`, so coverage stays auditable instead of silently short.

**Version tracking, for a model run rather than a download.** Every other extract in
this toolkit stamps a release date read out of the source file. There is no source file
here — the output is produced, not fetched — so a date would describe nothing useful.
What a rerun actually has to match is the *model*: the `pangolin` package version, the
weights it loaded (hashed, because the version string alone would not catch a swapped
file), the reference region the sequences were cut from, and the torch build and device.
The cell writes those to `data/pangolin_cftr.release.json` at run time and
`load_pangolin()` exposes them as `pangolin_release`.

This can only be recorded **while the model runs**. An extract built before the stamp
existed cannot have its provenance reconstructed afterwards — the honest answer for one
of those is "unknown", and that is what the loader reports. The cell says so explicitly
rather than inventing a plausible-looking stamp from whatever happens to be installed now.

License: Pangolin is **non-commercial** — cite Zeng & Li 2022 (PMID 35449021).

In [2]:
import re, json, hashlib, requests, numpy as np
from datetime import datetime, timezone

DATA_DIR = pathlib.Path.cwd().parent / "data"
PANGOLIN_TSV = DATA_DIR / "pangolin_cftr.csv"
PANGOLIN_RELEASE_JSON = DATA_DIR / "pangolin_cftr.release.json"
CFTR2_CSV = DATA_DIR / "cftr2_cftr.csv"
REF_FA = DATA_DIR / "cftr_region_grch38.fa"
SCOPE = "cftr2"                    # "cftr2" (REAL, ~4 min on GPU) or "curated" (DEMO, seconds)
DIST = 50                          # Pangolin's aggregation window, +/- d
MAX_EVENT = 100                    # skip ref/alt events bigger than this (score would be meaningless)
KNOWN_SPLICE = ["c.2988+1G>A", "c.2657+5G>A", "c.3718-2477C>T", "c.3140-26A>G", "c.1680-886A>G"]
_ACGT = re.compile(r"^[ACGT]+$")

if PANGOLIN_TSV.exists():
    print(f"already built -> {PANGOLIN_TSV.name} (delete to rebuild, or edit SCOPE above and rerun)")
    if not PANGOLIN_RELEASE_JSON.exists():
        print(f"  ! no {PANGOLIN_RELEASE_JSON.name} beside it -- this extract predates release\n"
              "    tracking, so what produced it cannot be established after the fact and\n"
              "    load_pangolin() will report pangolin_release as 'unknown'. To get a real\n"
              f"    stamp, delete {PANGOLIN_TSV.name} and rerun this cell.")
else:
    try:
        import torch
        from pangolin.model import Pangolin, L, W, AR
        from pkg_resources import resource_filename
    except ImportError as exc:
        raise ImportError(
            "Pangolin needs one extra install (no other notebook needs it):\n"
            '  pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils torch'
        ) from exc
    if not CFTR2_CSV.exists():
        raise FileNotFoundError(f"{CFTR2_CSV} missing -- run benchmark/01_cftr2.ipynb first "
                                 "(Pangolin needs its authoritative GRCh38 coordinates).")

    # one_hot_encode + compute_score are inlined verbatim from pangolin/pangolin.py
    # (Zeng & Li 2022) so we skip importing that module, whose own top-level
    # `import pyfastx, vcf` pulls in dependencies this notebook doesn't otherwise need.
    IN_MAP = np.asarray([[0, 0, 0, 0], [1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

    def one_hot_encode(seq, strand):
        seq = seq.upper().replace('A', '1').replace('C', '2').replace('G', '3').replace('T', '4').replace('N', '0')
        if strand == '+':
            seq = np.asarray(list(map(int, list(seq))))
        else:
            seq = np.asarray(list(map(int, list(seq[::-1]))))
            seq = (5 - seq) % 5
        return IN_MAP[seq.astype('int8')]

    def compute_score(ref_seq, alt_seq, strand, d, models):
        ref_seq = torch.from_numpy(np.expand_dims(one_hot_encode(ref_seq, strand).T, axis=0)).float()
        alt_seq = torch.from_numpy(np.expand_dims(one_hot_encode(alt_seq, strand).T, axis=0)).float()
        if torch.cuda.is_available():
            ref_seq, alt_seq = ref_seq.to("cuda"), alt_seq.to("cuda")
        pang = []
        for j in range(4):
            score = []
            for model in models[3 * j:3 * j + 3]:
                with torch.no_grad():
                    ref = model(ref_seq)[0][[1, 4, 7, 10][j], :].cpu().numpy()
                    alt = model(alt_seq)[0][[1, 4, 7, 10][j], :].cpu().numpy()
                    if strand == '-':
                        ref, alt = ref[::-1], alt[::-1]
                    l = 2 * d + 1
                    ndiff = np.abs(len(ref) - len(alt))
                    if len(ref) > len(alt):
                        alt = np.concatenate([alt[0:l // 2 + 1], np.zeros(ndiff), alt[l // 2 + 1:]])
                    elif len(ref) < len(alt):
                        alt = np.concatenate([alt[0:l // 2], np.max(alt[l // 2:l // 2 + ndiff + 1], keepdims=True), alt[l // 2 + ndiff + 1:]])
                    score.append(alt - ref)
            pang.append(np.mean(score, axis=0))
        pang = np.array(pang)
        loss = pang[np.argmin(pang, axis=0), np.arange(pang.shape[1])]
        gain = pang[np.argmax(pang, axis=0), np.arange(pang.shape[1])]
        return loss, gain

    def load_models():
        """Load the 12 bundled Pangolin models, recording each weight file's SHA-256 into
        WEIGHTS -- the version string alone wouldn't catch a swapped or truncated file."""
        dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        models = []
        for i in [0, 2, 4, 6]:
            for j in range(1, 4):
                m = Pangolin(L, W, AR)
                wpath = resource_filename("pangolin", "models/final.%s.%s.3.v2" % (j, i))
                WEIGHTS[pathlib.Path(wpath).name] = hashlib.sha256(
                    pathlib.Path(wpath).read_bytes()).hexdigest()
                m.load_state_dict(torch.load(wpath, map_location=dev))
                models.append(m.to(dev).eval())
        return models


    WEIGHTS = {}          # weight-file name -> sha256, filled in by load_models()

    def write_release(scope, n_scored, n_targets, region_header):
        """Stamp the MODEL RUN, not a download: what a rerun has to match to reproduce
        these scores. Only ever called on the path where the model actually ran."""
        try:
            from importlib.metadata import version as _pkgver
            pkg = _pkgver("pangolin")
        except Exception:
            pkg = "unknown"
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        resolved = (f"Pangolin pkg {pkg}, 12 bundled weight files "
                    f"(sha256 of models/final.*.3.v2 recorded below), torch {torch.__version__} on {dev}")
        PANGOLIN_RELEASE_JSON.write_text(json.dumps({
            "resolved_version": resolved,
            "pangolin_package_version": pkg,
            "model_weight_sha256": WEIGHTS,
            "torch_version": torch.__version__,
            "device": dev,
            "reference_region": region_header,
            "reference_region_file": REF_FA.name,
            "aggregation_window_bp": DIST,
            "max_event_bp": MAX_EVENT,
            "scope": scope,
            "scored": n_scored,
            "targets": n_targets,
            "run_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }, indent=2))
        print(f"release stamp -> {PANGOLIN_RELEASE_JSON.name}: {resolved}")

    def load_region():
        """Return (region_start_1based, sequence, fasta_header); auto-fetches + caches the
        ~215 kb CFTR slice from Ensembl's REST API the first time -- no whole-genome FASTA.
        The header names the assembly and span, so it goes into the release stamp."""
        if not REF_FA.exists():
            print("fetching CFTR reference region from Ensembl (one-time, ~215 kb)...")
            r = requests.get(
                "https://rest.ensembl.org/sequence/region/human/7:117465000..117680000",
                headers={"Content-Type": "text/x-fasta"}, timeout=30)
            r.raise_for_status()
            REF_FA.write_text(r.text)
        lines = REF_FA.read_text().splitlines()
        header = lines[0].lstrip(">")             # 'chromosome:GRCh38:7:117465000:117680000:1'
        r0 = int(header.split(":")[3])
        return r0, "".join(lines[1:]).upper(), header

    def pangolin_score(pos, ref, alt, r0, seq, models, d=DIST):
        start = (pos - r0) - (5000 + d)
        end = start + 10000 + 2 * d + len(ref)
        if start < 0 or end > len(seq):
            raise ValueError(f"outside the cached reference window (needs 7:{pos-5050}-{pos+5050})")
        window = seq[start:end]
        got = window[5000 + d: 5000 + d + len(ref)]
        if got != ref:
            raise ValueError(f"ref mismatch at 7:{pos} -- window has {got!r}, expected {ref!r}")
        alt_seq = window[:5000 + d] + alt + window[5000 + d + len(ref):]
        # Score in reference/plus-strand orientation -- matches how SpliceAI's precomputed
        # scores are reported and is validated against them (c.2988+1G>A: donor-loss 0.86
        # vs SpliceAI 0.99). CFTR is plus-strand; passing strand='-' would mis-score it.
        loss, gain = compute_score(window, alt_seq, "+", d, models)
        return round(float(max(gain.max(), -loss.min())), 4)

    cf = pd.read_csv(CFTR2_CSV)
    if SCOPE == "curated":
        cf = cf[cf["cdna_name"].isin(KNOWN_SPLICE)]
    cf = cf.copy()
    ref, alt = cf["grch38_ref"].astype(str), cf["grch38_alt"].astype(str)
    cf["skip_reason"] = None
    cf.loc[cf["grch38_pos"].isna(), "skip_reason"] = "no GRCh38 coordinates in CFTR2"
    bad = cf["skip_reason"].isna() & ~(ref.str.match(_ACGT) & alt.str.match(_ACGT))
    cf.loc[bad, "skip_reason"] = "allele is not plain ACGT"
    big = cf["skip_reason"].isna() & ((ref.str.len() > MAX_EVENT) | (alt.str.len() > MAX_EVENT))
    cf.loc[big, "skip_reason"] = f"event larger than {MAX_EVENT} bp"

    label = "REAL" if SCOPE == "cftr2" else "DEMO"      # label follows coverage, not the model
    print(f"scope={SCOPE} -> {len(cf):,} CFTR2 variants, source label '{label}'")
    r0, seq, region_header = load_region()
    models = load_models()

    rows, done = [], 0
    for _, v in cf.iterrows():
        score, reason = None, v["skip_reason"]
        pos = int(v["grch38_pos"]) if pd.notna(v["grch38_pos"]) else None
        r_, a_ = str(v["grch38_ref"]), str(v["grch38_alt"])
        if reason is None:
            try:
                score = pangolin_score(pos, r_, a_, r0, seq, models)
                done += 1
                if SCOPE == "curated" or done % 200 == 0:
                    print(f"  [{done:5,}] {str(v['cdna_name'])[:24]:24} pangolin={score}")
            except Exception as e:
                reason = str(e).split(" -- ")[0]
        rows.append({"cdna_name": v["cdna_name"], "legacy_name": v["legacy_name"],
                     "chrom": "7", "pos": pos, "ref": r_ if pos else None, "alt": a_ if pos else None,
                     "pangolin_score": score, "cftr2_class": v["cftr2_class"],
                     "source": label, "skip_reason": reason})

    out = pd.DataFrame(rows)
    out.to_csv(PANGOLIN_TSV, index=False)
    scored = out["pangolin_score"].notna()
    print(f"\nPangolin ({label}) written: {int(scored.sum()):,} scored / {len(out):,} targets "
          f"-> {PANGOLIN_TSV.relative_to(DATA_DIR.parent)}")
    write_release(SCOPE, int(scored.sum()), len(out), region_header)

already built -> pangolin_cftr.csv (delete to rebuild, or edit SCOPE above and rerun)
  ! no pangolin_cftr.release.json beside it -- this extract predates release
    tracking, so what produced it cannot be established after the fact and
    load_pangolin() will report pangolin_release as 'unknown'. To get a real
    stamp, delete pangolin_cftr.csv and rerun this cell.


## Example: the shared splice worked-example panel, scored by **Pangolin**

## 3 · Coverage, the panel, and how well the two models actually agree

The same fixed panel of famous CFTR **splice** variants runs through both splice
notebooks (tools/07 and tools/08), so you can follow one set of variants across the
series. The variant list is `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the **scoring
is shown inline below** so you can see exactly how Pangolin is joined onto it.

CFTR2's legacy names and coordinates are joined on but deliberately **not** printed —
its terms forbid republishing any portion of its content. Counts and correlations are
facts *about* the list rather than reproductions of it, so those are shown.

The last block is the claim this notebook opened with, measured: SpliceAI and Pangolin
are independent models, so where they agree, the agreement is worth something.

In [3]:
# Pangolin scores come from RUNNING the model (build cell above) over the CFTR2 list
# with correct CFTR2 coords. Tier with the published 0.5 / 0.2 cut-points.
pg = tk.load_pangolin()
HIGH, MOD = tk.THRESHOLDS['pangolin']['high'], tk.THRESHOLDS['pangolin']['moderate']
scored = pg[pg['pangolin_score'].notna()].copy()
scored['tier'] = scored['pangolin_score'].apply(
    lambda s: 'HIGH' if s >= HIGH else ('MODERATE' if s >= MOD else 'LOW'))
print(f"{len(scored):,} scored / {len(pg):,} CFTR2 targets | source: {pg['source'].unique().tolist()}")
print(f"release: {pg['pangolin_release'].iloc[0]}")
print(scored['tier'].value_counts().to_string())
if pg['pangolin_score'].isna().any():
    print("\nnot scored, by reason:")
    print(pg.loc[pg['pangolin_score'].isna(), 'skip_reason'].value_counts().to_string())

# The classic CF splice alleles — the validation set, recovered from the full run.
# CFTR2's legacy_name / coordinates are joined on but not printed (see above).
print("\nclassic CF splice alleles:")
print(scored[scored['cdna_name'].isin(tk.A2_KNOWN_CDNA)]
      [['cdna_name', 'pangolin_score', 'tier', 'source']].to_string(index=False))

# Do the two independent models agree? Measure it rather than assert it.
sp = tk.load_spliceai()
if set(sp['source'].unique()) == {'REAL'}:
    both = (scored.dropna(subset=['pos'])
            .assign(pos=lambda d: d['pos'].astype(int))
            .merge(sp[['pos', 'ref', 'alt', 'spliceai_ds_max']], on=['pos', 'ref', 'alt']))
    r = both['pangolin_score'].corr(both['spliceai_ds_max'])
    agree = ((both['pangolin_score'] >= HIGH) == (both['spliceai_ds_max'] >= HIGH)).mean()
    print(f"\nscored by BOTH Pangolin and SpliceAI: {len(both):,} CFTR2 variants")
    print(f"  pearson r                  : {r:.3f}")
    print(f"  same side of the 0.5 cut   : {agree:.1%}")
else:
    print("\n(SpliceAI extract not built -- run tools/07 to compare the two models)")

1,892 scored / 2,097 CFTR2 targets | source: ['REAL']
release: unknown (pre-dates release tracking; re-run the build cell)
tier
LOW         1518
HIGH         260
MODERATE     114

not scored, by reason:
skip_reason
no GRCh38 coordinates in CFTR2    204
event larger than 100 bp            1

classic CF splice alleles:
     cdna_name  pangolin_score     tier source
c.3718-2477C>T          0.3327 MODERATE   REAL
   c.2657+5G>A          0.8194     HIGH   REAL
  c.3140-26A>G          0.8120     HIGH   REAL
   c.2988+1G>A          0.8568     HIGH   REAL
 c.1680-886A>G          0.7057     HIGH   REAL



scored by BOTH Pangolin and SpliceAI: 1,728 CFTR2 variants
  pearson r                  : 0.968
  same side of the 0.5 cut   : 97.9%


## Key takeaways

1. **Pangolin** gives one 0-1 score with the **same** 0.5 / 0.2 thresholds as SpliceAI.
2. That single score is a **collapse**: it is `max(gain, |loss|)` over the ±50 bp window,
   and the direction is discarded before it reaches the CSV. When you need to know
   *whether a site is being created or destroyed*, read tools/07's four deltas — no
   amount of re-reading this extract will recover it.
3. SpliceAI + Pangolin **agreeing** is stronger evidence than either alone, and the cell
   above measures that agreement across every CFTR2 variant both models score rather than
   inferring it from five famous alleles.
4. The build cell produces **real** Pangolin scores locally, over the whole CFTR2
   list (1,892 of 2,097 scored, `source='REAL'`); `SCOPE = "curated"` keeps the 5-allele
   teaching run at `source='DEMO'`. The label follows coverage, not the model.
5. Running locally beats a precomputed release on **reach**: Pangolin scores 575 of the
   576 coordinate-carrying CFTR2 indels, against 417 for the SpliceAI extract. But reach
   is not usefulness — most of those extra rows are frameshifts, where "no splice impact"
   is correct and beside the point.
6. **Version** is a property of the model run, not a download date, so the stamp records
   the package version, the weight-file hashes, the reference region and the device —
   written while the model runs, because it cannot be reconstructed afterwards.
7. Correct citation: **Zeng & Li 2022, PMID 35449021**.

**This is the last notebook in the published series.** The live CADD API notebook and the
cross-tool benchmark over the whole CFTR2 list are written but held back pending the same
audit pass these notebooks have been through.